In [5]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

DATA_FILE = Path(
    "/workspaces/machine-learning-zoomcamp-homework/"
    "05-deployment/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

assert DATA_FILE.is_file(), f"Dataset not found: {DATA_FILE}"

df = pd.read_csv(DATA_FILE)

print("Dataset shape:", df.shape)
print(df["Churn"].value_counts())

Dataset shape: (7043, 21)
Churn
No     5174
Yes    1869
Name: count, dtype: int64


In [6]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"], errors="coerce"
).fillna(0)

df.columns = df.columns.str.lower().str.replace(" ", "_")

string_columns = df.select_dtypes(include="object").columns

for col in string_columns:
    df[col] = (
        df[col]
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )

df["churn"] = df["churn"].map({"no": 0, "yes": 1})

assert df["churn"].notna().all()
assert df["churn"].value_counts().to_dict() == {
    0: 5174, 1: 1869
}

print(df["churn"].value_counts())

churn
0    5174
1    1869
Name: count, dtype: int64


/tmp/ipykernel_4312/2811172734.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = df.select_dtypes(include="object").columns


In [7]:
df_train_full, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train_full = df_train_full.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train_full["churn"].values
y_test = df_test["churn"].values

print(df_train_full["churn"].value_counts())

churn
0    4113
1    1521
Name: count, dtype: int64


In [8]:
categorical = [
    "gender", "seniorcitizen", "partner", "dependents",
    "phoneservice", "multiplelines", "internetservice",
    "onlinesecurity", "onlinebackup", "deviceprotection",
    "techsupport", "streamingtv", "streamingmovies",
    "contract", "paperlessbilling", "paymentmethod"
]

numerical = [
    "tenure", "monthlycharges", "totalcharges"
]

In [9]:
def train(df, y, C=1.0):
    records = df[categorical + numerical].to_dict(
        orient="records"
    )

    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(records)

    model = LogisticRegression(
        solver="liblinear", C=C
    )
    model.fit(X, y)

    return dv, model


def predict(df, dv, model):
    records = df[categorical + numerical].to_dict(
        orient="records"
    )

    X = dv.transform(records)
    return model.predict_proba(X)[:, 1]

In [10]:
dv, model = train(df_train_full, y_train, C=0.5)

test_predictions = predict(df_test, dv, model)

auc = roc_auc_score(y_test, test_predictions)

print(f"Test AUC: {auc:.3f}")
print("Model features:", len(dv.get_feature_names_out()))

Test AUC: 0.858
Model features: 45


In [11]:
customer = {
    "customerid": "8879-zkjof",
    "gender": "female",
    "seniorcitizen": 0,
    "partner": "no",
    "dependents": "no",
    "tenure": 41,
    "phoneservice": "yes",
    "multiplelines": "no",
    "internetservice": "dsl",
    "onlinesecurity": "yes",
    "onlinebackup": "no",
    "deviceprotection": "yes",
    "techsupport": "yes",
    "streamingtv": "yes",
    "streamingmovies": "yes",
    "contract": "one_year",
    "paperlessbilling": "yes",
    "paymentmethod": "bank_transfer_(automatic)",
    "monthlycharges": 79.85,
    "totalcharges": 3320.75
}

In [13]:
# Cell 8 — Customer churn prediction

customer_df = pd.DataFrame([customer])

churn_probability = float(
    predict(customer_df, dv, model)[0]
)

print(f"Customer churn probability: {churn_probability:.6f}")
print(f"Customer churn percentage: {churn_probability * 100:.2f}%")

# Compare with the original reference
expected_probability = 0.059606

difference = abs(churn_probability - expected_probability)

print(f"Difference from reference: {difference:.6f}")

Customer churn probability: 0.061876
Customer churn percentage: 6.19%
Difference from reference: 0.002270


In [14]:
# Cell 9 — Save the trained model

import pickle

with open("churn-model.bin", "wb") as f:
    pickle.dump((dv, model), f)

print("Model saved successfully")

Model saved successfully


In [22]:
import subprocess

result = subprocess.run(
    ["uv", "--version"],
    capture_output=True,
    text=True
)

print(result.stdout or result.stderr)

uv 0.13.5 (x86_64-unknown-linux-gnu)



In [21]:
import sklearn

print("Current scikit-learn version:", sklearn.__version__)

Current scikit-learn version: 1.9.1


In [23]:
from pathlib import Path
import pickle

HW_DIR = Path(
    "/workspaces/machine-learning-zoomcamp-homework/"
    "2026-course-reference/cohorts/2026/homework/05-deployment"
)

with open(HW_DIR / "pipeline.bin", "rb") as f:
    pipeline = pickle.load(f)

lead_q3 = {
    "lead_source": "paid_ads",
    "industry": "technology",
    "employment_status": "employed",
    "location": "north_america",
    "number_of_courses_viewed": 2,
    "annual_income": 79276.0,
    "interaction_count": 4,
    "lead_score": 0.41
}

prob_q3 = float(pipeline.predict_proba([lead_q3])[0, 1])

print("Q3 answer:", round(prob_q3, 3))

Q3 answer: 0.533


/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator DictVectorizer from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.7.2 when using version 1.9.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/python/3.14.2/lib/python3.14/site-packages/sklearn/base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.9

In [25]:
import requests

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

response = requests.post(
    "http://localhost:9696/predict",
    json=lead_q4,
    timeout=30
)

response.raise_for_status()
result = response.json()

print("API response:", result)
print("Q4 answer:", round(result["conversion_probability"], 3))

API response: {'conversion_probability': 0.769799, 'conversion': True}
Q4 answer: 0.77


In [1]:
import requests

lead_q4 = {
    "lead_source": "organic_search",
    "industry": "technology",
    "employment_status": "employed",
    "location": "europe",
    "number_of_courses_viewed": 4,
    "annual_income": 80304.0,
    "interaction_count": 7,
    "lead_score": 0.74
}

response = requests.post(
    "http://localhost:9696/predict",
    json=lead_q4,
    timeout=30
)

response.raise_for_status()
result = response.json()

print("API response:", result)
print("Q4 answer:", round(result["conversion_probability"], 3))

API response: {'conversion_probability': 0.769799, 'conversion': True}
Q4 answer: 0.77
